# Popularity baseline: alignment end-to-end

This notebook is the first minimal train-eval loop skeleton:

1. build Agent4Rec/SimUSER-style interaction alignment tasks for several `1:m` candidate ratios;
2. split task frames into `X, y`;
3. fit `PopularityScorer` on train;
4. select a binary decision threshold on validation;
5. evaluate pointwise binary metrics on test.


In [ ]:
from __future__ import annotations

import sys
import time
from pathlib import Path

import pandas as pd


def _add_repo_root_to_path() -> None:
    for path in [Path.cwd(), *Path.cwd().parents]:
        if (path / "pyproject.toml").exists() and (path / "runners").exists():
            for import_path in (str(path), str(path / "src")):
                if import_path not in sys.path:
                    sys.path.insert(0, import_path)
            return
    raise RuntimeError("Could not find beyond-click-sim repo root")


_add_repo_root_to_path()

from runners.in_distribution.interaction_prediction.task_builders import (
    data_root,
    repo_root,
)

REPO_ROOT = repo_root()
DATA_ROOT = data_root()

pd.set_option("display.max_columns", 120)
print(f"repo_root={REPO_ROOT}")
print(f"data_root={DATA_ROOT}")


In [ ]:
from beyond_click_sim.data.canonical import CanonicalDataset
from beyond_click_sim.evaluation import (
    apply_threshold,
    binary_classification_metrics,
    find_best_threshold,
)
from beyond_click_sim.scorers import PopularityScorer
from beyond_click_sim.tasks import (
    AlignmentInteractionTaskBuilder,
    CappedUserInteractionCandidateSampler,
    MinUserInteractionsFilter,
    NonInteractionCandidateSampler,
    RandomFractionSplitter,
    split_xy,
)


In [ ]:
DATASETS = ("ml-1m", "steam")
VERSION = "v1"
SEED = 0
MIN_INTERACTIONS = 10
# Agent4Rec Table 1 uses interacted:unobserved ratios 1:m with m in {1, 2, 3, 9}.
NEGATIVE_RATIOS = (1, 2, 3, 9)
# Use ("per_positive", "agent4rec_cap20") to compare with the previous construction.
SAMPLER_PROTOCOLS = ("agent4rec_cap20",)
TOTAL_CANDIDATE_ITEMS = 20
TRAIN_FRACTION = 0.7
VAL_FRACTION = 0.1
TEST_FRACTION = 0.2
THRESHOLD_METRIC = "f1"


In [ ]:
def load_canonical_dataset(dataset_name: str) -> CanonicalDataset:
    root = DATA_ROOT / dataset_name / VERSION
    return CanonicalDataset(
        name=dataset_name,
        version=VERSION,
        root=root,
        users_path=root / "users.parquet",
        items_path=root / "items.parquet",
        interactions_path=root / "interactions.parquet",
        manifest_path=root / "manifest.json",
    )


def make_sampler(sampler_protocol: str, negative_ratio: int):
    if sampler_protocol == "per_positive":
        return NonInteractionCandidateSampler(
            negative_ratio=negative_ratio,
            seed=SEED,
        )
    if sampler_protocol == "agent4rec_cap20":
        return CappedUserInteractionCandidateSampler(
            negative_ratio=negative_ratio,
            total_items=TOTAL_CANDIDATE_ITEMS,
            seed=SEED,
        )
    raise ValueError(f"Unknown sampler protocol: {sampler_protocol!r}")


def build_alignment_task(
    dataset: CanonicalDataset,
    *,
    sampler_protocol: str,
    negative_ratio: int,
):
    builder = AlignmentInteractionTaskBuilder(
        name=(
            f"{dataset.name}-interaction-alignment-random-"
            f"popularity-{sampler_protocol}-m{negative_ratio}"
        ),
        dataset_filter=MinUserInteractionsFilter(min_interactions=MIN_INTERACTIONS),
        splitter=RandomFractionSplitter(
            train_fraction=TRAIN_FRACTION,
            val_fraction=VAL_FRACTION,
            test_fraction=TEST_FRACTION,
            seed=SEED,
            group_column="user_id",
        ),
        sampler=make_sampler(
            sampler_protocol=sampler_protocol,
            negative_ratio=negative_ratio,
        ),
    )
    return builder.build(dataset)


def candidate_summary(frame: pd.DataFrame, task) -> dict[str, int | float]:
    group_column = task.schema.candidate_group_column
    sampled_column = task.schema.sampled_column
    target_column = task.schema.target_column
    if group_column is None or sampled_column is None:
        raise ValueError("candidate and sampled columns are required")
    group_sizes = frame.groupby(group_column).size()
    positives = frame.groupby(group_column)[target_column].sum()
    sampled = frame.groupby(group_column)[sampled_column].sum()
    return {
        "candidate_sets": int(len(group_sizes)),
        "rows_min": int(group_sizes.min()),
        "rows_max": int(group_sizes.max()),
        "positives_min": float(positives.min()),
        "positives_max": float(positives.max()),
        "sampled_min": int(sampled.min()),
        "sampled_max": int(sampled.max()),
    }


In [ ]:
def run_popularity_alignment(
    dataset_name: str,
    *,
    sampler_protocol: str,
    negative_ratio: int,
) -> dict[str, object]:
    started = time.perf_counter()
    dataset = load_canonical_dataset(dataset_name)
    task = build_alignment_task(
        dataset,
        sampler_protocol=sampler_protocol,
        negative_ratio=negative_ratio,
    )
    build_seconds = time.perf_counter() - started

    target_column = task.schema.target_column
    X_train, y_train = split_xy(task.train, target_column=target_column)
    X_val, y_val = split_xy(task.val, target_column=target_column)
    X_test, y_test = split_xy(task.test, target_column=target_column)

    scorer = PopularityScorer(item_column="item_id").fit(X_train, y_train)

    val_scores = scorer.score(X_val)
    threshold_selection = find_best_threshold(
        y_val,
        val_scores,
        metric=THRESHOLD_METRIC,
    )
    threshold = float(threshold_selection["threshold"])
    val_pred = apply_threshold(val_scores, threshold)
    val_metrics = binary_classification_metrics(y_val, val_pred)

    test_scores = scorer.score(X_test)
    test_pred = apply_threshold(test_scores, threshold)
    test_metrics = binary_classification_metrics(y_test, test_pred)

    return {
        "dataset": dataset_name,
        "sampler_protocol": sampler_protocol,
        "negative_ratio": negative_ratio,
        "candidate_ratio": f"1:{negative_ratio}",
        "total_candidate_items": TOTAL_CANDIDATE_ITEMS
        if sampler_protocol == "agent4rec_cap20"
        else None,
        "task_name": task.name,
        "build_seconds": round(build_seconds, 3),
        "train_rows": int(len(task.train)),
        "val_rows": int(len(task.val)),
        "test_rows": int(len(task.test)),
        "users": int(task.manifest["users"]),
        "items": int(task.manifest["items"]),
        "val_candidate_summary": candidate_summary(task.val, task),
        "test_candidate_summary": candidate_summary(task.test, task),
        "threshold": threshold,
        "threshold_metric": threshold_selection["metric"],
        "threshold_metric_value": threshold_selection["metric_value"],
        "val_accuracy": val_metrics["accuracy"],
        "val_precision": val_metrics["precision"],
        "val_recall": val_metrics["recall"],
        "val_f1": val_metrics["f1"],
        "val_n_predicted_positive": val_metrics["n_predicted_positive"],
        "test_accuracy": test_metrics["accuracy"],
        "test_precision": test_metrics["precision"],
        "test_recall": test_metrics["recall"],
        "test_f1": test_metrics["f1"],
        "test_n_predicted_positive": test_metrics["n_predicted_positive"],
    }


In [ ]:
rows = []
for dataset_name in DATASETS:
    for sampler_protocol in SAMPLER_PROTOCOLS:
        for negative_ratio in NEGATIVE_RATIOS:
            print(f"Running {dataset_name}, {sampler_protocol}, m={negative_ratio}...")
            row = run_popularity_alignment(
                dataset_name,
                sampler_protocol=sampler_protocol,
                negative_ratio=negative_ratio,
            )
            rows.append(row)
            print(
                f"  done in {row['build_seconds']}s; "
                f"threshold={row['threshold']}; "
                f"test_f1={row['test_f1']:.4f}"
            )

results = pd.DataFrame(rows)
results


In [ ]:
metric_columns = [
    "dataset",
    "sampler_protocol",
    "negative_ratio",
    "candidate_ratio",
    "total_candidate_items",
    "build_seconds",
    "train_rows",
    "val_rows",
    "test_rows",
    "threshold",
    "threshold_metric",
    "threshold_metric_value",
    "val_accuracy",
    "val_precision",
    "val_recall",
    "val_f1",
    "val_n_predicted_positive",
    "test_accuracy",
    "test_precision",
    "test_recall",
    "test_f1",
    "test_n_predicted_positive",
]
results.loc[:, metric_columns]


In [ ]:
candidate_summaries = pd.DataFrame(
    [
        {
            "dataset": row["dataset"],
            "sampler_protocol": row["sampler_protocol"],
            "negative_ratio": row["negative_ratio"],
            "candidate_ratio": row["candidate_ratio"],
            "total_candidate_items": row["total_candidate_items"],
            "split": split,
            **row[f"{split}_candidate_summary"],
        }
        for row in rows
        for split in ("val", "test")
    ]
)
candidate_summaries
